# 03 — Exploratory Data Analysis
**Purpose:** Understand distributions, trends, and anomalies. Generate visual hypotheses for each of the three pillars before formal statistical testing.  
**Input:** `data/clean/*.csv`  
**Output:** Inline charts · `reports/03_eda_findings.md`  
**Next step:** `04_statistical_analysis.ipynb`

## 3.0 — Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

CLEAN_DIR  = Path('data/clean')
REPORT_DIR = Path('reports')
FIG_DIR    = Path('reports/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Consistent style
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (12, 5),
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('tab10')

# Load all clean tables
master  = pd.read_csv(CLEAN_DIR / 'results_enriched.csv', parse_dates=['date','dob'])
races   = pd.read_csv(CLEAN_DIR / 'races.csv')
lt      = pd.read_csv(CLEAN_DIR / 'lapTimes.csv')
pit     = pd.read_csv(CLEAN_DIR / 'pitStops.csv')
qual    = pd.read_csv(CLEAN_DIR / 'qualifying.csv')
ds      = pd.read_csv(CLEAN_DIR / 'driverStandings.csv')
cs      = pd.read_csv(CLEAN_DIR / 'constructorStandings.csv')

ERA_ORDER = ['Pre-V10', 'V10', 'V8', 'Hybrid']
ERA_COLORS = {'Pre-V10': '#9e9e9e', 'V10': '#2196F3', 'V8': '#FF9800', 'Hybrid': '#4CAF50'}

print('All tables loaded. Master shape:', master.shape)

## 3.1 — Dataset Overview

In [ ]:
print('=== Coverage ===')
print(f'Seasons: {master["year"].min()} – {master["year"].max()}')
print(f'Races: {master["raceId"].nunique():,}')
print(f'Drivers: {master["driverId"].nunique():,}')
print(f'Constructors: {master["constructorId"].nunique():,}')
print(f'Circuits: {master["circuitId"].nunique():,}')
print(f'Total race entries: {len(master):,}')
print(f'Total DNFs: {master["dnf"].sum():,} ({master["dnf"].mean()*100:.1f}%)')

# Entries per era
era_counts = master.groupby('era').size().reindex(ERA_ORDER)
print('\nEntries per era:')
print(era_counts)

## 3.2 — PILLAR A: Driver Alpha Exploration

In [ ]:
# --- 3.2.1  Grid-to-finish distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

g2f = master['grid_vs_finish'].dropna()
axes[0].hist(g2f, bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1, label='No change')
axes[0].set_title('Grid-to-finish delta distribution')
axes[0].set_xlabel('Places gained (positive) or lost (negative)')
axes[0].set_ylabel('Count')
axes[0].legend()

# --- 3.2.2  Top 20 drivers by avg places gained (min 50 races) ---
driver_perf = master[master['grid'] > 0].groupby(['driverId','full_name']).agg(
    avg_gain=('grid_vs_finish','mean'),
    races=('raceId','count')
).reset_index().query('races >= 50').sort_values('avg_gain', ascending=False).head(20)

axes[1].barh(driver_perf['full_name'], driver_perf['avg_gain'], color='steelblue')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_title('Avg places gained per race (min 50 starts)')
axes[1].set_xlabel('Avg grid-to-finish delta')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(FIG_DIR / '3.2.1_grid_to_finish.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 3.2.2  Qualifying delta between teammates ---
# Join qualifying with race year and constructor
qual_enriched = qual.merge(races[['raceId','year']], on='raceId', how='left')\
                    .merge(master[['raceId','driverId','constructorId','full_name']].drop_duplicates(),
                           on=['raceId','driverId'], how='left')

# For each race × constructor, find teammate pairs and compute Q1 delta
def teammate_deltas(df):
    rows = []
    for (race_id, cid), grp in df.groupby(['raceId','constructorId']):
        grp = grp.dropna(subset=['q1_ms'])
        if len(grp) == 2:
            d1, d2 = grp.iloc[0], grp.iloc[1]
            delta = d1['q1_ms'] - d2['q1_ms']   # positive = d1 slower
            rows.append({'raceId': race_id, 'constructorId': cid, 'year': d1['year'],
                         'driver_a': d1['full_name'], 'driver_b': d2['full_name'],
                         'q1_delta_ms': abs(delta),
                         'faster_driver': d1['full_name'] if delta > 0 else d2['full_name']})
    return pd.DataFrame(rows)

tm_deltas = teammate_deltas(qual_enriched)
print('Teammate qualifying pairs found:', len(tm_deltas))
print('Median Q1 delta (ms):', tm_deltas['q1_delta_ms'].median())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(tm_deltas['q1_delta_ms'] / 1000, bins=50, color='darkorange', edgecolor='white', linewidth=0.5)
ax.set_title('Teammate Q1 qualifying gap distribution')
ax.set_xlabel('Gap in seconds')
ax.set_ylabel('Race count')
plt.tight_layout()
plt.savefig(FIG_DIR / '3.2.2_qualifying_gaps.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 3.2.3  Points per race by driver (top 30 drivers, min 30 races) ---
ppr = master.groupby(['driverId','full_name']).agg(
    total_points=('points','sum'),
    races=('raceId','count')
).reset_index()
ppr['points_per_race'] = ppr['total_points'] / ppr['races']
ppr = ppr.query('races >= 30').sort_values('points_per_race', ascending=False).head(30)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(ppr['full_name'], ppr['points_per_race'], color='steelblue')
ax.set_title('Points per race — top 30 drivers (min 30 starts)')
ax.set_xlabel('Avg points per race')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / '3.2.3_ppr.png', bbox_inches='tight')
plt.show()

## 3.3 — PILLAR B: Pit Stop Strategy Exploration

In [ ]:
# --- 3.3.1  Pit stop duration distribution by year ---
pit_valid = pit[pit['is_valid_stop']].copy()
pit_valid = pit_valid.merge(races[['raceId','year','era']], on='raceId', how='left')

fig, ax = plt.subplots(figsize=(14, 5))
for era in ['V8', 'Hybrid']:    # only 2011-2017 coverage
    data = pit_valid[pit_valid['era'] == era]['duration_ms'] / 1000
    ax.hist(data, bins=60, alpha=0.6, label=era)
ax.set_title('Pit stop duration distribution by era (2011–2017)')
ax.set_xlabel('Duration (seconds)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '3.3.1_pit_durations.png', bbox_inches='tight')
plt.show()

print('Median pit stop duration (s):')
print(pit_valid.groupby('era')['duration_ms'].median() / 1000)

In [ ]:
# --- 3.3.2  Number of pit stops vs final position ---
stops_per_race = pit_valid.groupby(['raceId','driverId'])['stop'].max().reset_index()
stops_per_race.rename(columns={'stop': 'total_stops'}, inplace=True)

stops_vs_pos = stops_per_race.merge(
    master[['raceId','driverId','positionOrder','dnf','year']],
    on=['raceId','driverId'], how='left'
)
stops_vs_pos = stops_vs_pos[~stops_vs_pos['dnf']].copy()

fig, ax = plt.subplots(figsize=(10, 5))
bp_data = [stops_vs_pos[stops_vs_pos['total_stops'] == n]['positionOrder'].values
           for n in sorted(stops_vs_pos['total_stops'].unique())[:5]]
ax.boxplot(bp_data, labels=sorted(stops_vs_pos['total_stops'].unique())[:5])
ax.set_title('Finish position by number of pit stops (non-DNF finishers)')
ax.set_xlabel('Total pit stops in race')
ax.set_ylabel('Final position')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / '3.3.2_stops_vs_position.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 3.3.3  Lap-position before vs after pit stop (track position analysis) ---
# For each pit stop: get track position on lap N-1 and lap N+3
lt_pos = lt[['raceId','driverId','lap','position']].copy()

def position_change_around_pit(pit_row, lt_pos):
    r, d, lap = pit_row['raceId'], pit_row['driverId'], pit_row['lap']
    pre  = lt_pos[(lt_pos.raceId==r) & (lt_pos.driverId==d) & (lt_pos.lap==lap-1)]['position']
    post = lt_pos[(lt_pos.raceId==r) & (lt_pos.driverId==d) & (lt_pos.lap==lap+3)]['position']
    if len(pre) and len(post):
        return int(pre.iloc[0]) - int(post.iloc[0])   # positive = gained places
    return np.nan

# Sample 2000 stops for speed (remove for full run)
sample_pit = pit_valid.sample(min(2000, len(pit_valid)), random_state=42)
sample_pit['pos_change'] = sample_pit.apply(lambda r: position_change_around_pit(r, lt_pos), axis=1)

fig, ax = plt.subplots(figsize=(10, 4))
sample_pit['pos_change'].dropna().hist(bins=30, ax=ax, color='teal', edgecolor='white')
ax.axvline(0, color='red', linestyle='--')
ax.set_title('Position change 3 laps after pit stop (sample n=2000)')
ax.set_xlabel('Positions gained (positive = better)')
plt.tight_layout()
plt.savefig(FIG_DIR / '3.3.3_position_change_post_pit.png', bbox_inches='tight')
plt.show()

## 3.4 — PILLAR C: Reliability & DNF Exploration

In [ ]:
# --- 3.4.1  DNF rate by era ---
dnf_era = master.groupby('era').agg(
    total_entries=('resultId','count'),
    total_dnf=('dnf','sum')
).reindex(ERA_ORDER)
dnf_era['dnf_rate'] = dnf_era['total_dnf'] / dnf_era['total_entries']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(dnf_era.index, dnf_era['dnf_rate'] * 100,
              color=[ERA_COLORS[e] for e in dnf_era.index])
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=10)
ax.set_title('Overall DNF rate by engine era')
ax.set_ylabel('DNF rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig(FIG_DIR / '3.4.1_dnf_by_era.png', bbox_inches='tight')
plt.show()
print(dnf_era)

In [ ]:
# --- 3.4.2  DNF category breakdown by era ---
dnf_only = master[master['dnf'] == True].copy()
dnf_cat_era = dnf_only.groupby(['era','dnf_category']).size().unstack(fill_value=0).reindex(ERA_ORDER)
dnf_cat_era_pct = dnf_cat_era.div(dnf_cat_era.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
dnf_cat_era_pct.plot(kind='bar', stacked=True, ax=ax,
                     color=['#4CAF50','#F44336','#FF9800','#9E9E9E'])
ax.set_title('DNF type breakdown by era (% of all DNFs in era)')
ax.set_ylabel('%')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(loc='upper right', title='DNF type')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / '3.4.2_dnf_categories_by_era.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 3.4.3  Rolling 3-season mechanical DNF rate trend ---
mech_dnf_yr = master.groupby('year').agg(
    entries=('resultId','count'),
    mech_dnf=('dnf_category', lambda x: (x == 'Mechanical DNF').sum())
).reset_index()
mech_dnf_yr['mech_dnf_rate'] = mech_dnf_yr['mech_dnf'] / mech_dnf_yr['entries']
mech_dnf_yr['rolling3'] = mech_dnf_yr['mech_dnf_rate'].rolling(3, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(mech_dnf_yr['year'], mech_dnf_yr['mech_dnf_rate'], color='#9E9E9E', alpha=0.5, label='Annual')
ax.plot(mech_dnf_yr['year'], mech_dnf_yr['rolling3'], color='#F44336', linewidth=2, label='3-season rolling avg')
for era, (start, end) in {'V10': (1995,2005), 'V8': (2006,2013), 'Hybrid': (2014,2018)}.items():
    ax.axvspan(start, end, alpha=0.08, color=ERA_COLORS[era], label=era)
ax.set_title('Mechanical DNF rate over time (1950–2018)')
ax.set_ylabel('Mechanical DNF rate')
ax.set_xlabel('Season')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '3.4.3_mech_dnf_trend.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 3.4.4  Constructor reliability heatmap (post-1990 for readability) ---
rel_matrix = master[master['year'] >= 1990].groupby(['constructor_name','year']).agg(
    entries=('resultId','count'),
    dnf=('dnf','sum')
).reset_index()
rel_matrix['reliability'] = 1 - rel_matrix['dnf'] / rel_matrix['entries']

# Pivot: constructors with >= 5 seasons
pivot = rel_matrix.pivot(index='constructor_name', columns='year', values='reliability')
active_constr = pivot.notna().sum(axis=1)
pivot_filtered = pivot[active_constr >= 5].fillna(0)

fig, ax = plt.subplots(figsize=(18, max(6, len(pivot_filtered) * 0.35)))
sns.heatmap(pivot_filtered, cmap='RdYlGn', vmin=0.5, vmax=1.0, ax=ax,
            linewidths=0.2, linecolor='white', cbar_kws={'label': 'Reliability index'})
ax.set_title('Constructor reliability index by season (1990–2018)')
ax.set_xlabel('Season')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(FIG_DIR / '3.4.4_constructor_reliability_heatmap.png', bbox_inches='tight')
plt.show()

## 3.5 — Cross-Pillar: Championship Context

In [ ]:
# --- 3.5.1  Points accumulation trajectory — select seasons ---
SELECT_YEARS = [2008, 2012, 2016]   # close championship battles

for yr in SELECT_YEARS:
    yr_data = master[master['year'] == yr].copy()
    yr_data = yr_data.merge(races[['raceId','round']], on='raceId', how='left')
    yr_data.sort_values('round', inplace=True)
    yr_data['cum_points'] = yr_data.groupby('driverId')['points'].cumsum()

    # Top 5 drivers by end-of-season points
    top5 = yr_data.groupby('driverId')['points'].sum().nlargest(5).index
    yr_top = yr_data[yr_data['driverId'].isin(top5)]

    fig, ax = plt.subplots(figsize=(12, 5))
    for drv_id, grp in yr_top.groupby('driverId'):
        name = grp['full_name'].iloc[0]
        ax.plot(grp['round'], grp['cum_points'], marker='o', markersize=3, label=name)
    ax.set_title(f'{yr} Championship — cumulative points (top 5)')
    ax.set_xlabel('Round')
    ax.set_ylabel('Cumulative points')
    ax.legend(loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'3.5.1_championship_{yr}.png', bbox_inches='tight')
    plt.show()

## 3.6 — EDA Summary & Hypotheses for Statistical Testing

In [ ]:
eda_findings = """
# EDA Findings — Key Hypotheses for 04_statistical_analysis

## Pillar A — Driver Alpha
- H1: Some drivers consistently outperform their constructors' average
  (measured by points_per_race vs constructor avg)
- H2: Qualifying pace delta between teammates correlates with race finish delta
- H3: Grid-to-finish improvement is driver-specific, not random

## Pillar B — Pit Strategy
- H4: 2-stop strategy is associated with worse average finish than 1-stop in hybrid era
- H5: Pit stop duration < median correlates with position gain in the 3 laps after stop
- H6: Pit lap timing (early vs late in stint) influences final race position

## Pillar C — Reliability
- H7: Mechanical DNF rate in the Hybrid era is NOT higher than V10 era (counter-narrative)
- H8: Variance in reliability (std dev of DNF rate) differs significantly across constructors
- H9: Points lost to DNF is concentrated in top-5 contenders (not backmarkers)
"""

(REPORT_DIR / '03_eda_findings.md').write_text(eda_findings)
print('EDA findings written to reports/03_eda_findings.md')
print(eda_findings)